In [2]:
import pandas as pd
from mp_api.client import MPRester
from matminer.featurizers.conversions import StrToComposition
from matminer.featurizers.composition import ElementProperty

apikey = "4Fxu8kCP9i4QlEKWL4gUCKY8WfTujzM8"

fields = [
    "material_id", "formula_pretty", "bulk_modulus", "shear_modulus", 
    "density", "band_gap", "formation_energy_per_atom", 
    "homogeneous_poisson", "symmetry", "energy_above_hull"
]

with MPRester(apikey) as mpr:
    docs = mpr.materials.summary.search(has_props=["elasticity"], is_stable=True, fields=fields)
    print(f"Initial number:{len(docs)} materials with elasticity data.")
    
    results = []
    for doc in docs:

        b = doc.bulk_modulus.get('vrh') if isinstance(doc.bulk_modulus, dict) else getattr(doc.bulk_modulus, 'vrh', None)
        g = doc.shear_modulus.get('vrh') if isinstance(doc.shear_modulus, dict) else getattr(doc.shear_modulus, 'vrh', None)
        p_ratio = doc.homogeneous_poisson
        c_system = getattr(doc.symmetry, 'crystal_system', "Unknown")
        ehull = doc.energy_above_hull if doc.energy_above_hull is not None else 999
        
        if b and g and b > 0 and g > 0 and ehull <= 0:
            k = g / b
            hv = max(0, 2 * (pow(k**2 * g, 0.585)) - 3)
            
            results.append({
                "id": doc.material_id,
                "formula": doc.formula_pretty,
                "vickers_hardness_gpa": hv,
                "density": doc.density,
                "band_gap": doc.band_gap,
                "formation_energy": doc.formation_energy_per_atom,
                "poissons_ratio": p_ratio,
                "crystal_system": str(c_system),
                "bulk_modulus": b,
                "shear_modulus": g,
                "ehull": ehull
            })

df = pd.DataFrame(results)
print(f"Reduction 1: {len(df)} materials  after filtering for stability and valid moduli.")

df = df.dropna()
print(f"Reduction 2:  {len(df)} materials after dropping rows with missing values (NaN)")

stc = StrToComposition()
df = stc.featurize_dataframe(df, "formula") 

ep_feat = ElementProperty(
    data_source="magpie",
    features=["AtomicWeight", "Number", "Electronegativity", "MeltingT", "CovalentRadius"],
    stats=["mean", "avg_dev", "range"] 
)

df = ep_feat.featurize_dataframe(df, col_id="composition")
print(f"Final size: {len(df)}")

df.to_csv("mp_hardness_dataset4.csv", index=False)

Retrieving SummaryDoc documents:   0%|          | 0/6087 [00:00<?, ?it/s]

Initial number:6087 materials with elasticity data.
Reduction 1: 6087 materials  after filtering for stability and valid moduli.
Reduction 2:  6087 materials after dropping rows with missing values (NaN)


StrToComposition:   0%|          | 0/6087 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/6087 [00:00<?, ?it/s]

Final size: 6087
